[![Labellerr](https://storage.googleapis.com/labellerr-cdn/%200%20Labellerr%20template/notebook.webp)](https://www.labellerr.com)

# **Electronic Chips Components Segmentation**

---

[![labellerr](https://img.shields.io/badge/Labellerr-BLOG-black.svg)](https://www.labellerr.com/blog/<BLOG_NAME>)
[![Youtube](https://img.shields.io/badge/Labellerr-YouTube-b31b1b.svg)](https://www.youtube.com/@Labellerr)
[![Github](https://img.shields.io/badge/Labellerr-GitHub-green.svg)](https://github.com/Labellerr/Hands-On-Learning-in-Computer-Vision)

## Project Overview  
This project leverages **YOLOv11x-seg** to perform high-precision **instance segmentation** of electronic components on a Raspberry Pi 4.  
Unlike standard object detection that uses bounding boxes, this system generates **pixel-level masks** to accurately identify the exact boundaries of 9 distinct hardware ports and chips.

To ensure professional-grade clarity and accuracy, the system incorporates:

- **Instance Segmentation**  
  Precisely isolates the shape of components such as USB ports, HDMI slots, and GPIO pins.

- **Retina Mask Technology**  
  Processes masks at native resolution to eliminate blocky edges, especially on small components.

- **Micro-Labeling System**  
  Uses optimized font scaling to display clear labels without obstructing the hardware view.

- **High-Resolution Inference**  
  Designed for dense PCB environments where components are tightly packed.

---

## Real-World Applications  

### Automated Quality Assurance (QA)  
Used in manufacturing pipelines to verify that all components (USB ports, capacitors, SoC) are correctly placed and soldered before packaging.

### Educational AR Overlays  
Enables augmented reality applications that help students identify hardware components in real-time using live camera feeds.

### E-Waste Sorting & Recycling  
Supports robotic systems in detecting and extracting valuable components such as CPUs or wireless modules from discarded circuit boards.

### Automated Inventory Management  
Quickly scans hardware units to identify specific models (e.g., Raspberry Pi 4 vs Pi 3) based on port layout and component structure.

---

## Key Features  

- **9-Class Specialized Vocabulary**  
  Trained to distinguish between visually similar components such as Micro HDMI and USB-C ports.

- **Retina-Grade Segmentation**  
  High-definition mask rendering ensures accurate detection of fine structures like GPIO pins.

- **Clean UI Inference**  
  Displays only colored masks with minimal text labels, removing unnecessary bounding boxes for clarity.

- **State-of-the-Art Architecture**  
  Built on the YOLOv11x backbone to achieve high accuracy (mAP) in complex and dense electronic environments.

---

## Annotate your Custom dataset using Labellerr

 ***1. Visit the [Labellerr](https://www.labellerr.com/?utm_source=githubY&utm_medium=social&utm_campaign=github_clicks) website and click **“Sign Up”**.*** 

 ***2. After signing in, create your workspace by entering a unique name.***

 ***3. Navigate to your workspace’s API keys page (e.g., `https://<your-workspace>.labellerr.com/workspace/api-keys`) to generate your **API Key** and **API Secret**.***

 ***4. Store the credentials securely, and then use them to initialise the SDK or API client with `api_key`, `api_secret`.*** 



## Import Libraries

This section imports all the required libraries used throughout the project for computer vision, visualization, deep learning, and structured coding.


In [1]:
!pip install ultralytics opencv-python matplotlib cv2

ERROR: Could not find a version that satisfies the requirement cv2 (from versions: none)

[notice] A new release of pip is available: 25.1.1 -> 26.0.1
[notice] To update, run: C:\Python313\python.exe -m pip install --upgrade pip
ERROR: No matching distribution found for cv2


In [2]:
!git clone https://github.com/Labellerr/yolo_finetune_utils.git

Cloning into 'yolo_finetune_utils'...


## 🎞️ Random Frame Extraction from Video

Extracts a fixed number of high-quality frames from one or more videos to create an image dataset for annotation and training.

### 🔹 Purpose
- Convert raw manufacturing videos into individual image frames  
- Perform random sampling to avoid frame bias  
- Prepare data for annotation and YOLO training  


In [ ]:
from yolo_finetune_utils.frame_extractor import extract_random_frames

extract_random_frames(
    paths=['Electronic Chips.mp4'],
    total_images=50,
    out_dir="dataset_frames",
    jpg_quality=100,
    seed=42
)

[✓] Extracted 20 frames to folder: dataset_frames


## 📥 Download Annotations from Labellerr

After completing data labeling on the **Labellerr** platform, export the annotations in **COCO JSON format**.

Download the COCO JSON file from the Labellerr website and upload it into this project workspace to use it for further dataset preparation and training.

This COCO JSON file will be used in the next steps for:
- Frame–annotation alignment
- COCO → YOLO format conversion
- Model training and evaluation


# COCO to YOLO Format Conversion

Converts COCO-style segmentation annotations to YOLO segmentation dataset format.  
- Requires: `annotation.json` and images in `frames_output` directory.
- Output: Generated YOLO dataset folder.
- Parameters: allows train/val split, shuffling, and verbose mode.


In [2]:
from yolo_finetune_utils.coco_yolo_converter.seg_converter import coco_to_yolo_converter

coco_to_yolo_converter(
    json_path="928704a1-c37b-4546-94a3-19ec49a9d6fc.json",
    images_dir="dataset_frames",
    output_dir="yoloDataset",
    use_split=True,
    train_ratio=0.8,
    val_ratio=0.1,
    test_ratio=0.1,
    shuffle=True,
    verbose=True
)


Conversion complete. Stats: {'train': 16, 'val': 2, 'test': 2}


{'stats': {'train': 16, 'val': 2, 'test': 2}, 'output_dir': 'yoloDataset'}

# Load and Train YOLO Segmentation Model

Loads the YOLO segmentation model and trains it using the converted YOLO dataset.
- Data: Path to YOLO-style `data.yaml`
- Parameters: epochs, image size, batch size, device, dataloader workers, experiment name.


In [ ]:

model = YOLO('yolo11x-seg.pt') 

results = model.train(
    data='/kaggle/working/data.yaml',
    epochs=150,
    imgsz=640,
    batch=16,
    project='/kaggle/working/pi_components',
    name='train_run_2',
    device=0,
    
    degrees=15.0,    
    fliplr=0.5,      
    flipud=0.5 
)      

# Inference Pipeline Overview

This script serves as the **inference pipeline** for the Raspberry Pi 4 component detection project.  
It automates the process of identifying hardware components using a custom-trained segmentation model.

---

## Key Functional Steps  

### Model Loading  
Initializes the **YOLOv11x-seg** architecture using the trained weights file (`best.pt`).  
These weights contain the learned features required to detect and segment the 9 Raspberry Pi components.

---

### High-Resolution Masking  
Enables `retina_masks=True` to generate segmentation masks at the original video resolution.  
This avoids downsampling and ensures **sharp, pixel-accurate boundaries** for small and dense components.

---

### Visual Optimization  
Configures the output to:
- Display **class labels only**
- Hide **bounding boxes**

This keeps the hardware clearly visible while still providing identification information.

---

### Memory-Efficient Streaming  
Uses `stream=True` to process the video **frame-by-frame**.  
This approach minimizes GPU memory usage and prevents crashes in constrained environments like Kaggle.

---

### Automated Export  
The `save=True` flag enables automatic saving of results.  
The processed frames are compiled into a final `.avi` video file and stored in the `predict` directory.

---

In [ ]:
from ultralytics import YOLO

model = YOLO("/kaggle/working/pi_components/train_run_2/weights/best.pt") 

video_path = "/kaggle/input/datasets/aaryanaggarwal5040/electrobestvid/Electronic Chips.mp4" 

print(f"Starting inference on {video_path}...")
results = model.predict(
    source=video_path,
    save=True, 
    conf=0.2,
    show_boxes=True,    
    show_labels=True,    
    show_conf=False,
    retina_masks=True,
    stream=True,
    line_width=1         
)

for frame_result in results:
    pass 

print("\nInference complete!")
print("Look for your newly generated video inside the 'runs/segment/predict/' folder.")

---

## 👨‍💻 About Labellerr's Hands-On Learning in Computer Vision

Thank you for exploring this **Labellerr Hands-On Computer Vision Cookbook**! We hope this notebook helped you learn, prototype, and accelerate your vision projects.  
Labellerr provides ready-to-run Jupyter/Colab notebooks for the latest models and real-world use cases in computer vision, AI agents, and data annotation.

---
## 🧑‍🔬 Check Our Popular Youtube Videos

Whether you're a beginner or a practitioner, our hands-on training videos are perfect for learning custom model building, computer vision techniques, and applied AI:

- [How to Fine-Tune YOLO on Custom Dataset](https://www.youtube.com/watch?v=pBLWOe01QXU)  
  Step-by-step guide to fine-tuning YOLO for real-world use—environment setup, annotation, training, validation, and inference.
- [Build a Real-Time Intrusion Detection System with YOLO](https://www.youtube.com/watch?v=kwQeokYDVcE)  
  Create an AI-powered system to detect intruders in real time using YOLO and computer vision.
- [Finding Athlete Speed Using YOLO](https://www.youtube.com/watch?v=txW0CQe_pw0)  
  Estimate real-time speed of athletes for sports analytics.
- [Object Counting Using AI](https://www.youtube.com/watch?v=smsjBBQcIUQ)  
  Learn dataset curation, annotation, and training for robust object counting AI applications.
---

## 🎦 Popular Labellerr YouTube Videos

Level up your skills and see video walkthroughs of these tools and notebooks on the  
[Labellerr YouTube Channel](https://www.youtube.com/@Labellerr/videos):

- [How I Fixed My Biggest Annotation Nightmare with Labellerr](https://www.youtube.com/watch?v=hlcFdiuz_HI) – Solving complex annotation for ML engineers.
- [Explore Your Dataset with Labellerr's AI](https://www.youtube.com/watch?v=LdbRXYWVyN0) – Auto-tagging, object counting, image descriptions, and dataset exploration.
- [Boost AI Image Annotation 10X with Labellerr's CLIP Mode](https://www.youtube.com/watch?v=pY_o4EvYMz8) – Refine annotations with precision using CLIP mode.
- [Boost Data Annotation Accuracy and Efficiency with Active Learning](https://www.youtube.com/watch?v=lAYu-ewIhTE) – Speed up your annotation workflow using Active Learning.

> 👉 **Subscribe** for Labellerr's deep learning, annotation, and AI tutorials, or watch videos directly alongside notebooks!

---

## 🤝 Stay Connected

- **Website:** [https://www.labellerr.com/](https://www.labellerr.com/)
- **Blog:** [https://www.labellerr.com/blog/](https://www.labellerr.com/blog/)
- **GitHub:** [Labellerr/Hands-On-Learning-in-Computer-Vision](https://github.com/Labellerr/Hands-On-Learning-in-Computer-Vision)
- **LinkedIn:** [Labellerr](https://in.linkedin.com/company/labellerr)
- **Twitter/X:** [@Labellerr1](https://x.com/Labellerr1)

*Happy learning and building with Labellerr!*
